# Notebook 02 · Preprocessing & Data Cleaning
**Input :** `F:\mmd\data\processed\reviews\*.parquet` + `meta\*.parquet`  
**Output:** `F:\mmd\data\cleaned\reviews_clean.parquet`  
           `F:\mmd\data\cleaned\meta_clean.parquet`  
           `F:\mmd\data\cleaned\sessions.parquet` ← user → sorted item sequence  

---
### Pipeline
```
reviews/          meta/
  │  lazy scan      │  lazy scan
  ▼                 ▼
dedup            dedup parent_asin
  │              parse price → float
filter verified_purchase=True
  │
filter rating >= 1 (all keep, implicit feedback)
  │
k-core filtering  (user ≥5, item ≥5 interactions)
  │                        ↑ lặp đến hội tụ
  ▼
build sessions   (group by user_id, sort timestamp)
  │
temporal split   train / val / test
  ▼
lưu parquet
```

In [1]:
import psutil
from tqdm import tqdm

def ram_usage():
    vm = psutil.virtual_memory()
    return f"RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)"
print(f"{ram_usage()}")

RAM: 6.3/7.9 GB (80%)


## 0 · Imports & config

In [2]:
import os, gc, sys, logging
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import psutil
from tqdm.auto import tqdm

import numpy as np
from pathlib import Path


ROOT_DIR      = Path(r"F:\amazon_data") 
REVIEW_PATH   = ROOT_DIR / "data" / "json" / "review_Home_and_Kitchen.jsonl"
META_PATH     = ROOT_DIR / "data" / "json" / "meta_Home_and_Kitchen.jsonl"

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
TEMP_DIR      = ROOT_DIR / "data" / "processed" / "_temp_chunks"

for d in [PROCESSED_DIR, SAMPLE_DIR, FIGURES_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    
CHUNK_SIZE           = 200_000
MIN_TEXT_LENGTH      = 10
MIN_REVIEWS_PER_USER = 5
MIN_REVIEWS_PER_ITEM = 5
RANDOM_SEED          = 42

np.random.seed(RANDOM_SEED)

print("=== CẤU HÌNH & THÔNG TIN FILE ===")
print(f"   Chunk size : {CHUNK_SIZE:,} dòng/lần")

if REVIEW_PATH.exists():
    print(f"   Review file: {REVIEW_PATH.stat().st_size/(1024**3):.2f} GB")
else:
    print(f"   Review file: Không tìm thấy tại {REVIEW_PATH}")

if META_PATH.exists():
    print(f"   Meta file  : {META_PATH.stat().st_size/(1024**3):.2f} GB")
else:
    print(f"   Meta file  : Không tìm thấy tại {META_PATH}")

print(f"   RAM: {ram_usage()}")

=== CẤU HÌNH & THÔNG TIN FILE ===
   Chunk size : 200,000 dòng/lần
   Review file: 29.25 GB
   Meta file  : 10.98 GB
   RAM: RAM: 6.3/7.9 GB (80%)


## 1 · Load reviews (lazy)

In [3]:
gc.collect()

0

In [5]:
import json
import gc
import psutil
import shutil
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from tqdm.auto import tqdm

# Đảm bảo đã khai báo thư viện logging
import logging
logging.basicConfig(format="%(asctime)s  %(levelname)-8s  %(message)s",
                    level=logging.INFO, datefmt="%H:%M:%S")
log = logging.getLogger("preprocess")

log.info("Bước 1/3: Đọc JSONL và xả liên tục ra ổ cứng để ép RAM...")

TEMP_DIR = PROCESSED_DIR / "_temp_chunks"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# Xóa file rác từ lần chạy trước (nếu có)
for f in TEMP_DIR.glob("chunk_*.parquet"):
    f.unlink()

CHUNK_SIZE = 500_000
current_chunk = []
chunk_id = 0
total_raw = 0

# Đọc file line-by-line
with open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    pbar = tqdm(f, desc="Processing JSONL", unit=" lines")
    
    for line in pbar:
        total_raw += 1
        try:
            data = json.loads(line.strip())
        except json.JSONDecodeError:
            continue
            
        # Lọc verified ngay lập tức (Bỏ qua rác)
        if not data.get("verified_purchase", False):
            continue
            
        u = data.get("user_id")
        it = data.get("parent_asin")
        r = data.get("rating")
        ts = data.get("timestamp")
        
        if not all([u, it, r, ts]):
            continue

        current_chunk.append({
            "user_id": u,
            "parent_asin": it,
            "rating": r,
            "timestamp": ts
        })
        
        # Đủ 500k dòng -> Xử lý & Xả ra ổ cứng
        if len(current_chunk) >= CHUNK_SIZE:
            df_chunk = pd.DataFrame(current_chunk)
            
            # Xóa trùng lặp cục bộ
            df_chunk = df_chunk.sort_values("timestamp", ascending=False)
            df_chunk = df_chunk.drop_duplicates(subset=["user_id", "parent_asin"], keep="first")
            
            chunk_path = TEMP_DIR / f"chunk_{chunk_id:04d}.parquet"
            df_chunk.to_parquet(chunk_path, index=False)
            
            chunk_id += 1
            current_chunk = [] 
            del df_chunk
            gc.collect()
            
            pbar.set_postfix({'Chunks': chunk_id, 'RAM': f"{psutil.virtual_memory().percent}%"})

# Xả nốt phần dư cuối cùng
if current_chunk:
    df_chunk = pd.DataFrame(current_chunk)
    df_chunk = df_chunk.sort_values("timestamp", ascending=False)
    df_chunk = df_chunk.drop_duplicates(subset=["user_id", "parent_asin"], keep="first")
    df_chunk.to_parquet(TEMP_DIR / f"chunk_{chunk_id:04d}.parquet", index=False)
    del df_chunk, current_chunk
    gc.collect()

log.info("Bước 2/3: Gộp luồng các chunk bằng ParquetWriter ...")

chunk_files = sorted(TEMP_DIR.glob("chunk_*.parquet"))
output_path = PROCESSED_DIR / "review_clean_temp.parquet"

writer = None
for chunk_file in tqdm(chunk_files, desc="Merging chunks"):
    table = pq.read_table(chunk_file)
    
    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema, compression='snappy')
        
    writer.write_table(table)
    del table
    gc.collect()

if writer:
    writer.close()

# Xóa thư mục chunk tạm
shutil.rmtree(TEMP_DIR)

log.info("Bước 3/3: Bắt đầu load file đã gộp vào RAM...")
df = pd.read_parquet(output_path, engine="pyarrow", dtype_backend="pyarrow")
log.info(f"Đã load {len(df):,} dòng. Đang sắp xếp (Sort) timestamp in-place...")
df.sort_values("timestamp", ascending=False, inplace=True)
gc.collect()
log.info("Sort xong! Đang xóa trùng lặp (Drop Duplicates)...")
# Tiếp tục dùng inplace=True
df.drop_duplicates(subset=["user_id", "parent_asin"], keep="first", inplace=True)
gc.collect()

log.info("Xóa trùng lặp xong! Đang format ngày tháng...")
df.reset_index(drop=True, inplace=True)
df["dt"] = pd.to_datetime(df["timestamp"], unit='ms')
if output_path.exists():
    output_path.unlink()

print("\n=== KẾT QUẢ XỬ LÝ HOÀN TẤT ===")
print(f"  Tổng dòng đọc gốc : {total_raw:,}")
print(f"  Dòng giữ lại      : {len(df):,}")
print(f"  Date range        : {df['dt'].min().date()} → {df['dt'].max().date()}")


07:54:50  INFO      Bước 1/3: Đọc JSONL và xả liên tục ra ổ cứng để ép RAM...


Processing JSONL: 0 lines [00:00, ? lines/s]

08:05:36  INFO      Bước 2/3: Gộp luồng các chunk bằng ParquetWriter ...


Merging chunks:   0%|          | 0/126 [00:00<?, ?it/s]

08:06:49  INFO      Bước 3/3: Bắt đầu load file đã gộp vào RAM...
08:08:10  INFO      Đã load 62,201,078 dòng. Đang sắp xếp (Sort) timestamp in-place...
08:14:31  INFO      Sort xong! Đang xóa trùng lặp (Drop Duplicates)...
08:20:46  INFO      Xóa trùng lặp xong! Đang format ngày tháng...



=== KẾT QUẢ XỬ LÝ HOÀN TẤT ===
  Tổng dòng đọc gốc : 67,409,944
  Dòng giữ lại      : 62,195,107
  Date range        : 2000-02-26 → 2023-09-13


## 2 · Clean Reviews
### 2.1 Load + dedup + filter verified

### 2.2 K-core filtering
Lặp lọc đến khi hội tụ: user ≥ 5 tương tác **và** item ≥ 5 tương tác.

In [7]:
import gc
from tqdm.auto import tqdm

def kcore_filter(df: pd.DataFrame, min_u: int, min_i: int) -> pd.DataFrame:
    """
    Lặp lọc user/item có ít tương tác đến khi kích thước không đổi.
    Thường hội tụ sau 3-5 vòng.
    """
    prev_len = -1
    iteration = 0
    
    # Khởi tạo thanh tiến trình không giới hạn số vòng lặp
    pbar = tqdm(desc="K-core Filtering", unit=" iter")
    
    while len(df) != prev_len:
        prev_len = len(df)
        iteration += 1
        
        # filter items
        item_counts = df["parent_asin"].value_counts()
        df = df[df["parent_asin"].isin(item_counts[item_counts >= min_i].index)]
        
        # filter users
        user_counts = df["user_id"].value_counts()
        df = df[df["user_id"].isin(user_counts[user_counts >= min_u].index)]
        
        # Ép gom rác RAM ngay lập tức sau các thao tác sinh bộ nhớ đệm lớn
        del item_counts, user_counts
        gc.collect()
        
        # Cập nhật thanh tiến trình và thông số lên màn hình
        pbar.update(1)
        pbar.set_postfix({
            'Rows': f"{len(df):,}",
            'Users': f"{df['user_id'].nunique():,}",
            'Items': f"{df['parent_asin'].nunique():,}",
            'RAM': ram() # Dùng hàm ram() đã định nghĩa ở trên
        })
        
    pbar.close()
    print(f"  Converged after {iteration} iterations.")
    return df.reset_index(drop=True)


print(f"Before k-core: {len(df):,} rows | "
      f"{df['user_id'].nunique():,} users | "
      f"{df['parent_asin'].nunique():,} items")

df = kcore_filter(df, MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS)

print(f"\nFinal      : {len(df):,} rows | "
      f"{df['user_id'].nunique():,} users | "
      f"{df['parent_asin'].nunique():,} items")
print(psutil.virtual_memory())

KeyboardInterrupt: 

## 3 · Clean Meta
Parse price, dedup `parent_asin`, filter chỉ giữ item có trong reviews.

In [ ]:
import re

log.info("Loading meta...")
META_COLS = ["parent_asin", "title", "main_category", "average_rating",
             "rating_number", "price", "store"]

# Đọc file meta từ file đơn
df_meta = pd.read_parquet(META_FILE, columns=META_COLS)
print(f"Loaded meta: {len(df_meta):,} rows  |  {ram()}")

# ── 3.1 Dedup parent_asin ─────────────────────────────────────────────
df_meta = df_meta.drop_duplicates(subset=["parent_asin"], keep="first").reset_index(drop=True)
print(f"After dedup: {len(df_meta):,} items")

# ── 3.2 Parse price → float ───────────────────────────────────────────
def parse_price(s) -> float:
    """'$12.99' → 12.99  |  'None'/'' → NaN  |  '10.99 - 20.99' → mean"""
    if pd.isna(s) or str(s).strip() in ("", "None", "null"):
        return float("nan")
    nums = re.findall(r"[\d]+(?:\.[\d]+)?", str(s))
    if not nums:
        return float("nan")
    vals = [float(n) for n in nums]
    return round(sum(vals) / len(vals), 2)   # range → lấy mean

df_meta["price_usd"] = df_meta["price"].apply(parse_price)
price_null = df_meta["price_usd"].isna().sum()
print(f"Price parsed: {len(df_meta)-price_null:,} valid | {price_null:,} NaN")
print(f"Price range : ${df_meta['price_usd'].min():.2f} – ${df_meta['price_usd'].quantile(0.99):.2f} (p99)")

# ── 3.3 Chỉ giữ item có trong reviews sau k-core ─────────────────────
valid_items = set(df["parent_asin"].unique())
df_meta = df_meta[df_meta["parent_asin"].isin(valid_items)].reset_index(drop=True)
print(f"After inner join: {len(df_meta):,} meta items  ({len(valid_items):,} unique in reviews)")
print(ram())

## 4 · Build Item Vocab
Map `parent_asin` (string) → `item_idx` (int) — cần thiết cho Item2Vec và GRU4Rec.

In [ ]:
# ── item vocab ────────────────────────────────────────────────────────
# Sắp xếp theo frequency giảm dần → idx nhỏ = item phổ biến hơn
item_freq  = df["parent_asin"].value_counts()   # đã sort descending
item2idx   = {item: idx for idx, item in enumerate(item_freq.index)}
idx2item   = {idx: item for item, idx in item2idx.items()}
N_ITEMS    = len(item2idx)

# ── user vocab ────────────────────────────────────────────────────────
user_freq  = df["user_id"].value_counts()
user2idx   = {u: i for i, u in enumerate(user_freq.index)}
N_USERS    = len(user2idx)

# Gán idx vào df
df["item_idx"] = df["parent_asin"].map(item2idx)
df["user_idx"] = df["user_id"].map(user2idx)

# Gán vào meta
df_meta["item_idx"] = df_meta["parent_asin"].map(item2idx)

print(f"Vocab size  : {N_ITEMS:,} items  |  {N_USERS:,} users")

# Lưu vocab
import json
vocab_dir = ROOT_DIR / "data" / "vocab"
vocab_dir.mkdir(exist_ok=True)
with open(vocab_dir / "item2idx.json", "w") as f:
    json.dump(item2idx, f)
with open(vocab_dir / "idx2item.json", "w") as f:
    json.dump({str(k): v for k, v in idx2item.items()}, f)
print(f"Vocab saved → {vocab_dir}")

## 5 · Build User Sessions
Group by `user_id`, sort theo `timestamp` tăng dần → ra sequence `[item_idx_0, item_idx_1, ...]`.

In [ ]:
log.info("Building sessions...")

# Sort theo (user, time)
df_sorted = df.sort_values(["user_idx", "timestamp"], ascending=True)

# Group → list of item indices per user
sessions = (
    df_sorted.groupby("user_idx", sort=False)["item_idx"]
    .apply(list)
    .reset_index()
    .rename(columns={"item_idx": "item_seq"})
)

# Cắt sequence quá dài (giữ MAX_SEQ_LEN cái mới nhất)
sessions["item_seq"] = sessions["item_seq"].apply(
    lambda seq: seq[-MAX_SEQ_LEN:] if len(seq) > MAX_SEQ_LEN else seq
)
sessions["seq_len"] = sessions["item_seq"].apply(len)

print(f"Total sessions : {len(sessions):,}")
print(f"Sequence length:")
print(sessions["seq_len"].describe().to_string())
print(ram())

## 6 · Temporal Train / Val / Test Split
**Leave-One-Out (LOO) temporal:**  
- `test`  = item **cuối cùng** trong sequence mỗi user  
- `val`   = item **áp cuối**  
- `train` = tất cả phần còn lại  

Đây là chuẩn phổ biến nhất trong session-based recommendation.

In [ ]:
# Chỉ giữ user có seq_len >= 3 (cần ít nhất 1 train + 1 val + 1 test)
sessions = sessions[sessions["seq_len"] >= 3].reset_index(drop=True)
print(f"Sessions with len>=3: {len(sessions):,}")

# ── LOO split ─────────────────────────────────────────────────────────
sessions["train_seq"] = sessions["item_seq"].apply(lambda s: s[:-2])
sessions["val_item"]  = sessions["item_seq"].apply(lambda s: s[-2])
sessions["test_item"] = sessions["item_seq"].apply(lambda s: s[-1])

# Thống kê
total_interactions = sessions["seq_len"].sum()
train_interactions = sessions["train_seq"].apply(len).sum()
print(f"\nSplit summary:")
print(f"  train interactions : {train_interactions:,}  ({train_interactions/total_interactions*100:.1f}%)")
print(f"  val  interactions  : {len(sessions):,}  ({len(sessions)/total_interactions*100:.1f}%)")
print(f"  test interactions  : {len(sessions):,}  ({len(sessions)/total_interactions*100:.1f}%)")
print(f"  total              : {total_interactions:,}")

## 7 · Save outputs

In [ ]:
# ── 7.1 reviews_clean.parquet ─────────────────────────────────────────
clean_cols = ["user_idx", "user_id", "item_idx", "parent_asin",
              "rating", "timestamp", "dt"]
df[clean_cols].to_parquet(CLEANED_DIR / "reviews_clean.parquet",
                           index=False, engine="pyarrow")
print(f"Saved reviews_clean.parquet  ({len(df):,} rows)")

# ── 7.2 meta_clean.parquet ────────────────────────────────────────────
df_meta.to_parquet(CLEANED_DIR / "meta_clean.parquet",
                   index=False, engine="pyarrow")
print(f"Saved meta_clean.parquet     ({len(df_meta):,} rows)")

# ── 7.3 sessions.parquet ─────────────────────────────────────────────
# Lưu dạng: user_idx | train_seq (list) | val_item (int) | test_item (int)
import pickle
sessions_save = sessions[["user_idx", "train_seq", "val_item",
                           "test_item", "seq_len"]].copy()

# Parquet không lưu được list of int natively → dùng pickle cho sessions
with open(CLEANED_DIR / "sessions.pkl", "wb") as f:
    pickle.dump(sessions_save, f)
print(f"Saved sessions.pkl           ({len(sessions_save):,} users)")

# Cũng lưu dạng flat parquet để dễ query
# Flatten: mỗi (user, position) là 1 dòng
records = []
for row in tqdm(sessions_save.itertuples(), total=len(sessions_save), desc="Flatten"):
    for pos, item in enumerate(row.train_seq):
        records.append((row.user_idx, pos, item, "train"))
    records.append((row.user_idx, len(row.train_seq), row.val_item, "val"))
    records.append((row.user_idx, len(row.train_seq)+1, row.test_item, "test"))

df_flat = pd.DataFrame(records, columns=["user_idx", "position", "item_idx", "split"])
df_flat.to_parquet(CLEANED_DIR / "interactions_flat.parquet",
                   index=False, engine="pyarrow")
print(f"Saved interactions_flat.parquet  ({len(df_flat):,} rows)")
del records, df_flat; gc.collect()

# ── 7.4 Stats summary ─────────────────────────────────────────────────
print("\n" + "="*50)
print("  PREPROCESSING COMPLETE")
print("="*50)
print(f"  Users          : {N_USERS:>10,}")
print(f"  Items          : {N_ITEMS:>10,}")
print(f"  Interactions   : {len(df):>10,}")
print(f"  Sessions       : {len(sessions_save):>10,}")
print(f"  Avg seq len    : {sessions_save['seq_len'].mean():>10.2f}")
print(f"  Sparsity       : {1 - len(df)/(N_USERS*N_ITEMS):>10.6f}")
print("="*50)
print(ram())

---
## Notes

| Quyết định | Lý do |
|---|---|
| Dedup giữ review mới nhất | Nếu user review item 2 lần, thông tin mới hơn có giá trị hơn |
| Filter `verified_purchase=True` | Loại spam/bot reviews, giữ tín hiệu chất lượng |
| K-core(5,5) lặp đến hội tụ | Đảm bảo mỗi user/item đủ data để học embedding |
| LOO temporal split | Chuẩn đánh giá phổ biến nhất, không data leakage |
| Cắt sequence > 200 | GRU4Rec tốn VRAM nếu sequence quá dài |
| Lưu cả pkl + flat parquet | pkl cho model training (list), parquet cho EDA/query |

**Output files:**
```
F:\mmd\data\cleaned\
    reviews_clean.parquet
    meta_clean.parquet
    sessions.pkl
    interactions_flat.parquet
F:\mmd\data\vocab\
    item2idx.json
    idx2item.json
```
**→ Notebook 03:** EDA sâu trên cleaned data